# SE-TransNet V4: SEED-IV Execution Pipeline

### Prerequisites:
1. **Hardware:** Set Accelerator to **GPU T4 x2** (or P100).
2. **Internet:** Turn **ON** (needed to clone the repo and install pip packages).
3. **Data:** Ensure your private `seed-iv-raw` dataset is attached to this notebook.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/idhawal/SEEDIV-TransNet.git
%cd SEEDIV-TransNet
!ls

In [ ]:
# 2. Install dependencies
!pip install -q torch torchvision einops scipy numpy pyyaml tqdm scikit-learn tensorboard mne
!pip install -q -r requirements.txt || true

In [ ]:
# 3. Link the dataset into the path the repo expects
import os
os.makedirs('data', exist_ok=True)

# Symlink Kaggle's read-only input into a writable local path
!ln -sfn /kaggle/input/seed-iv-raw/eeg_raw_data data/eeg_raw_data
!ls data/eeg_raw_data

In [ ]:
# 4. Preprocess the raw .mat files
# Note: Output goes to /kaggle/working/ because it is persistent and writable.
!python preprocess_seed4.py \
    --raw-path /kaggle/input/seed-iv-raw/eeg_raw_data \
    --save-path /kaggle/working/processed

In [ ]:
# 5. Train (Subject-Dependent Protocol)
!python train_sd.py \
    --config configs/config_sd.yaml \
    --data-path /kaggle/working/processed

In [ ]:
# 6. Train (Cross-Subject / LOSO Protocol)
!python train_cs.py \
    --config configs/config_cs.yaml \
    --data-path /kaggle/working/processed

## 💡 Important Kaggle Gotchas

| Issue | Fix |
|---|---|
| `/kaggle/input` is read-only | Always write your outputs, models, and `.npy` files to `/kaggle/working/`. |
| 9-hour session limit | Use `--epochs` reasonably; checkpoint frequently to `/kaggle/working/`. |
| 20 GB working dir cap | Delete intermediate `.npy` files if you run out of space. |
| Notebook dies on close | Click **Save Version → Save & Run All (Commit)** to run headless in the background. |
| Reproducibility | Pin `torch==2.x` matching your local CUDA in `requirements.txt`. |

---

### 💾 Optional: Persist preprocessed data as a Kaggle Dataset
Preprocessing takes time. After Cell 4 finishes once:
1. **Save Version** the notebook.
2. Go to the output panel → **New Dataset from Notebook Output**.
3. Attach that new dataset to future notebooks and completely skip the preprocessing step!